# XGBoost — Walmart Store Sales Forecasting

MLflow experiment: `XGBoost_Training`.
Runs: `XGBoost_Cleaning`, `XGBoost_Feature_Selection`, `XGBoost_CrossValidation`,
`XGBoost_HPO`, `XGBoost_Final`.

Features come from the shared `features.build_features` (one-hot encoded for XGBoost).
Metric: WMAE, holiday weeks weighted 5x. Validation: expanding window, horizon = 39 weeks.


## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install mlflow dagshub kaggle xgboost optuna scikit-learn pandas pyarrow
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("..")
print("IN_COLAB =", IN_COLAB)

In [ ]:
import os, mlflow
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/karev23/ML-final.mlflow"
if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

mlflow.set_experiment("XGBoost_Training")
print("tracking:", mlflow.get_tracking_uri())

## Imports and data

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

train, test, features, stores, sample = load_raw("data")

# merged frames keep Weekly_Sales on train (needed for lag/history features)
train_merged = merge_all(train, features, stores).reset_index(drop=True)
raw_train = train[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
raw_test = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
y = train["Weekly_Sales"].to_numpy(dtype=float)
dates = train["Date"].to_numpy()
print("train_merged:", train_merged.shape, "| raw_test:", raw_test.shape)

In [ ]:
# build one fold: train features from its own history, val features from the
# fold-train history only (so no future sales leak), then align columns
def build_fold(tr_mask, va_mask):
    tr = train_merged[tr_mask].reset_index(drop=True)
    va = train_merged[va_mask].reset_index(drop=True)
    Xtr = build_features(tr, sales_history_df=None, encode_categoricals=True).astype("float32")
    Xva = build_features(va.drop(columns=["Weekly_Sales"]), sales_history_df=tr,
                         encode_categoricals=True)
    Xva = Xva.reindex(columns=Xtr.columns, fill_value=0).astype("float32")
    ytr = tr["Weekly_Sales"].to_numpy(float)
    yva = va["Weekly_Sales"].to_numpy(float)
    wtr = np.where(tr["IsHoliday"].astype(bool), 5.0, 1.0)
    hva = va["IsHoliday"].astype(bool).to_numpy()
    return Xtr, ytr, wtr, Xva, yva, hva

# precompute the 3 folds once, then only the model changes during CV/HPO
FOLDS = [build_fold(tr, va) for tr, va in expanding_window_folds(dates, n_splits=3, horizon=39)]
print("folds:", [(f[0].shape, f[3].shape) for f in FOLDS])

def cv_wmae(params, clip_zero=True):
    scores = []
    for Xtr, ytr, wtr, Xva, yva, hva in FOLDS:
        m = xgb.XGBRegressor(**params)
        m.fit(Xtr, ytr, sample_weight=wtr)
        p = m.predict(Xva)
        if clip_zero:
            p = np.clip(p, 0, None)
        scores.append(wmae(yva, p, hva))
    return scores, float(np.mean(scores))

## Run: XGBoost_Cleaning

Markdown/NaN handling is done inside `build_features`. This run compares clipping negative
predictions at 0 against leaving them, and logs the choice.


In [ ]:
# L1 objective because the metric is (weighted) absolute error
BASE_PARAMS = dict(
    objective="reg:absoluteerror",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    n_jobs=-1,
    random_state=42,
)

with mlflow.start_run(run_name="XGBoost_Cleaning"):
    _, mean_noclip = cv_wmae(BASE_PARAMS, clip_zero=False)
    _, mean_clip = cv_wmae(BASE_PARAMS, clip_zero=True)
    mlflow.log_param("markdown_nan_fill", "0 + missing flags (in build_features)")
    mlflow.log_param("negatives", "clip predictions at 0")
    mlflow.log_metric("wmae_cv_noclip", mean_noclip)
    mlflow.log_metric("wmae_cv_clip", mean_clip)
    print(f"CV WMAE  no-clip={mean_noclip:,.1f}   clip@0={mean_clip:,.1f}")

CLIP_ZERO = mean_clip <= mean_noclip
print("CLIP_ZERO =", CLIP_ZERO)

## Run: XGBoost_Feature_Selection

Fit once, read the importances, drop zero-importance columns and re-score to check whether
the smaller feature set keeps the same WMAE.


In [ ]:
Xtr, ytr, wtr, Xva, yva, hva = FOLDS[-1]

m = xgb.XGBRegressor(**BASE_PARAMS)
m.fit(Xtr, ytr, sample_weight=wtr)
imp = (pd.DataFrame({"feature": Xtr.columns, "importance": m.feature_importances_})
         .sort_values("importance", ascending=False).reset_index(drop=True))

keep = imp.loc[imp["importance"] > 0, "feature"].tolist()
m2 = xgb.XGBRegressor(**BASE_PARAMS)
m2.fit(Xtr[keep], ytr, sample_weight=wtr)

def _score(model, X):
    p = model.predict(X)
    return wmae(yva, np.clip(p, 0, None) if CLIP_ZERO else p, hva)

wmae_all = _score(m, Xva)
wmae_keep = _score(m2, Xva[keep])

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    imp.to_csv("xgb_feature_importance.csv", index=False)
    mlflow.log_artifact("xgb_feature_importance.csv")
    mlflow.log_param("n_features_all", len(Xtr.columns))
    mlflow.log_param("n_features_kept", len(keep))
    mlflow.log_metric("wmae_all_features", wmae_all)
    mlflow.log_metric("wmae_kept_features", wmae_keep)
    print(f"features {len(Xtr.columns)} -> {len(keep)} | WMAE {wmae_all:,.1f} -> {wmae_keep:,.1f}")

print(imp.head(15).to_string(index=False))

## Run: XGBoost_CrossValidation

In [ ]:
with mlflow.start_run(run_name="XGBoost_CrossValidation"):
    fold_wmae, mean_wmae = cv_wmae(BASE_PARAMS, clip_zero=CLIP_ZERO)
    mlflow.log_params({k: BASE_PARAMS[k] for k in
                       ["objective", "n_estimators", "learning_rate", "max_depth"]})
    mlflow.log_param("holiday_sample_weight", 5)
    for i, w in enumerate(fold_wmae):
        mlflow.log_metric(f"wmae_fold_{i}", w)
    mlflow.log_metric("wmae_cv_mean", mean_wmae)
    mlflow.log_metric("wmae_cv_std", float(np.std(fold_wmae)))
    print("per-fold WMAE:", [round(w, 1) for w in fold_wmae], "| mean:", round(mean_wmae, 1))

## Run: XGBoost_HPO

Optuna search over the main parameters; each trial is logged as a nested run.


In [ ]:
import optuna

N_TRIALS = 15
FIXED = dict(objective="reg:absoluteerror", n_jobs=-1, random_state=42)

def objective(trial):
    params = dict(
        FIXED,
        n_estimators=trial.suggest_int("n_estimators", 300, 900, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        max_depth=trial.suggest_int("max_depth", 5, 12),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        _, mean_wmae = cv_wmae(params, clip_zero=CLIP_ZERO)
        mlflow.log_params(params)
        mlflow.log_metric("wmae_cv_mean", mean_wmae)
    return mean_wmae

with mlflow.start_run(run_name="XGBoost_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_params(study.best_params)
    mlflow.log_metric("wmae_cv_best", study.best_value)
    print("best WMAE:", round(study.best_value, 1))
    print("best params:", study.best_params)

BEST_PARAMS = dict(FIXED, **study.best_params)

## Run: XGBoost_Final

Fit the raw->prediction pipeline on the full training set and log it. `code_paths` ships the
modules so it reloads from the registry without the repo. This is what `model_inference.ipynb` loads.


In [ ]:
from mlflow.models import infer_signature
from pipeline import make_pipeline

final_pipe = make_pipeline(xgb.XGBRegressor(**BEST_PARAMS), features, stores, clip_zero=CLIP_ZERO)
final_pipe.fit(raw_train, y)

test_pred = final_pipe.predict(raw_test)  # runs on the raw test frame
print("predicted test rows:", test_pred.shape, "| sample:", np.round(test_pred[:5], 1))

with mlflow.start_run(run_name="XGBoost_Final"):
    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("clip_zero", CLIP_ZERO)
    _, mean_wmae = cv_wmae(BEST_PARAMS, clip_zero=CLIP_ZERO)
    mlflow.log_metric("wmae_cv_mean", mean_wmae)
    sig = infer_signature(raw_test, test_pred)
    mlflow.sklearn.log_model(
        final_pipe,
        artifact_path="model",
        signature=sig,
        code_paths=["pipeline.py", "features.py", "dataloader.py"],
        input_example=raw_test.head(3),
    )
    print("logged pipeline | CV WMAE:", round(mean_wmae, 1))

make_submission(raw_test, test_pred, "submission_xgboost.csv")
print("wrote submission_xgboost.csv")

## Results

- CV WMAE: ___  | Kaggle public LB WMAE: ___
- Top features: ___
- clip@0 helped: ___
- feature selection changed WMAE: ___
- vs LightGBM: ___
